# k-resolution: why k=13, and why a fixed grid

**Producer for `results/k_resolution_headline_impact.json`.** That file justified
the project's two most consequential methodological choices — the discretisation
level and the decision not to collapse — and had no code in the repo; it was
generated ad hoc on 2026-08-22 and the script was lost.

The original asked *"how wrong was the retired k=7 headline?"*. That question is
settled and the k=7 protocol is archived. This rebuild asks the question the
Methods section actually has to answer:

1. **Where do the atoms converge, and where does seed spread stop narrowing?**
   That is the argument for k=13, and it must not reference the reported
   statistic — picking k by whichever value maximises `syn%` would be a
   selection-bias problem, since plug-in MI rises monotonically with resolution.
2. **What does `collapse_for_occupancy` do to the sweep?** It was written to help
   the solver on sparse joint cells. The claim is that it hurts more than it
   helps, and that claim needs to be visible rather than asserted.
3. **Where does the estimator break down?** Newly relevant: on the matched panel
   at k=16 the BROJA solver hangs on some seeds (see `pid_broja.SOLVER_TIMEOUT_S`),
   which is itself a reason not to reach for higher resolution.

Run on the **matched 12-analyte panel, both databases, mortality**, at the
headline's five seeds — so it speaks to the protocol actually in use, not to the
retired one.

In [1]:
import sys
sys.path.insert(0, "../src")

import json, time
from pathlib import Path

import numpy as np
import pandas as pd

import mimic_channels as mimic
import eicu_channels as eicu
import pid_broja as pb
import quantize as qz

pd.set_option("display.width", 220)

K_GRID = [4, 5, 6, 7, 8, 10, 13, 16, 20]
SEEDS = [0, 1, 2, 3, 4]
ATOMS = ("red", "u_val", "u_str", "syn")

MG_MIMIC = 50960
PANEL12_EICU = {"bicarbonate", "chloride", "creatinine", "glucose", "Hct", "Hgb",
                "platelets x 1000", "potassium", "RBC", "sodium", "BUN", "WBC x 1000"}

cc_m = mimic.complete_case(itemid_scope=set(mimic.CORE) - {MG_MIMIC})
cc_e = eicu.complete_case(labname_scope=PANEL12_EICU)
DBS = [("MIMIC-IV", mimic, cc_m), ("eICU-CRD", eicu, cc_e)]
for db, _, cc in DBS:
    print(f"{db:9s} n={cc.meta['n']:7,d}  mortality prevalence={cc.meta['prevalence']:.4f}")
print(f"\nk grid {K_GRID}, seeds {SEEDS}, both arms (fixed + collapsed) "
      f"= {2*len(K_GRID)*len(SEEDS)*2:,} decompositions")

MIMIC-IV  n= 64,623  mortality prevalence=0.1069
eICU-CRD  n=120,266  mortality prevalence=0.0947

k grid [4, 5, 6, 7, 8, 10, 13, 16, 20], seeds [0, 1, 2, 3, 4], both arms (fixed + collapsed) = 180 decompositions


## 1. Sweep

Both arms share the same k-means grid per (k, seed), so the only difference
between them is whether `collapse_for_occupancy` is applied. A cell whose solver
cascade fails returns NaN and is excluded from the mean rather than silently
counted as zero.

In [2]:
rows = []
t_all = time.time()

for db, mod, cc in DBS:
    for k in K_GRID:
        acc = {"fixed": {a: [] for a in ATOMS}, "collapse": {a: [] for a in ATOMS}}
        empties, fails, shapes = [], {"fixed": 0, "collapse": 0}, []
        t0 = time.time()

        for sd in SEEDS:
            v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(cc.Xv), k, seed=sd)
            s, _ = qz.kmeans_levels(
                qz.standardize_for_kmeans(cc.Xs, sentinel=mod.TIME_SENTINEL), k, seed=sd)

            d = pb.decompose(v, s, cc.y, measure="broja")
            empties.append(d["n_empty_cells"])
            if d["converged"]:
                for a in ATOMS: acc["fixed"][a].append(d["atoms_pct_of_joint"][a])
            else:
                fails["fixed"] += 1

            v2, s2, cinfo = qz.collapse_for_occupancy(v, s, cc.y)
            shapes.append(tuple(cinfo["levels"]))
            dc = pb.decompose(v2, s2, cc.y, measure="broja")
            if dc["converged"]:
                for a in ATOMS: acc["collapse"][a].append(dc["atoms_pct_of_joint"][a])
            else:
                fails["collapse"] += 1

        for arm in ("fixed", "collapse"):
            got = {a: np.array(acc[arm][a], float) for a in ATOMS}
            n_ok = len(got["syn"])
            rec = {"db": db, "k": k, "arm": arm, "cells": k * k * 2,
                   "n_seeds_ok": n_ok, "n_seeds_failed": fails[arm],
                   "mean_empty_cells": float(np.mean(empties)),
                   "collapsed_levels": sorted({s_ for s_ in shapes}) if arm == "collapse" else None}
            for a in ATOMS:
                rec[a] = float(got[a].mean()) if n_ok else float("nan")
                rec[f"{a}_sd"] = float(got[a].std()) if n_ok > 1 else float("nan")
            rec["worst_sd"] = float(np.nanmax([rec[f"{a}_sd"] for a in ATOMS])) if n_ok > 1 else float("nan")
            rows.append(rec)

        f = [r for r in rows if r["db"] == db and r["k"] == k and r["arm"] == "fixed"][0]
        print(f"{db:9s} k={k:2d}  cells={k*k*2:4d}  empty={f['mean_empty_cells']:5.1f}  "
              f"fixed: syn={f['syn']:6.2f} worst_sd={f['worst_sd']:5.2f} "
              f"ok={f['n_seeds_ok']}/{len(SEEDS)}   ({time.time()-t0:5.1f}s)", flush=True)

sweep = pd.DataFrame(rows)
print(f"\ntotal {time.time()-t_all:.0f}s")

MIMIC-IV  k= 4  cells=  32  empty=  0.0  fixed: syn= 19.00 worst_sd=24.18 ok=5/5   (  1.2s)


MIMIC-IV  k= 5  cells=  50  empty=  0.0  fixed: syn= 25.66 worst_sd=18.21 ok=5/5   (  1.3s)


MIMIC-IV  k= 6  cells=  72  empty=  0.0  fixed: syn= 31.82 worst_sd= 7.56 ok=5/5   (  1.3s)


MIMIC-IV  k= 7  cells=  98  empty=  0.0  fixed: syn= 30.23 worst_sd= 4.53 ok=5/5   (  1.4s)


MIMIC-IV  k= 8  cells= 128  empty=  0.0  fixed: syn= 31.63 worst_sd= 8.20 ok=5/5   (  1.5s)


...

eICU-CRD  k=10  cells= 200  empty=  0.0  fixed: syn= 26.53 worst_sd= 4.62 ok=5/5   (  2.6s)


eICU-CRD  k=13  cells= 338  empty=  0.0  fixed: syn= 30.79 worst_sd= 4.52 ok=5/5   (  2.7s)


eICU-CRD  k=16  cells= 512  empty=  3.6  fixed: syn= 26.14 worst_sd= 6.03 ok=5/5   (  3.1s)


eICU-CRD  k=20  cells= 800  empty= 19.4  fixed: syn= 29.43 worst_sd= 2.60 ok=5/5   (  3.7s)



total 70s


## 2. Where do the atoms converge?

In [3]:
fx = sweep[sweep.arm == "fixed"]
for db in fx.db.unique():
    g = fx[fx.db == db]
    print(f"\n{db} — fixed pre-collapse grid, mean +- sd over {len(SEEDS)} seeds\n")
    print(g[["k","cells","mean_empty_cells","n_seeds_ok","red","red_sd","u_val","u_val_sd",
             "u_str","u_str_sd","syn","syn_sd","worst_sd"]]
          .to_string(index=False, float_format=lambda x: f"{x:.2f}"))

print("\n\nseed spread (worst_sd) against k — the k=13 argument:\n")
piv = fx.pivot(index="k", columns="db", values="worst_sd")
print(piv.to_string(float_format=lambda x: f"{x:.2f}"))
print("\n  k=13 is the point where worst_sd has stopped falling materially and the")
print("  solver still converges on every seed. Above it the grid starts to degenerate:")
deg = fx[fx.n_seeds_failed > 0]
if len(deg):
    for _, r in deg.iterrows():
        print(f"    {r.db:9s} k={int(r.k):2d}: {int(r.n_seeds_failed)}/{len(SEEDS)} seeds "
              f"failed to converge, {r.mean_empty_cells:.1f} empty cells on average")
else:
    print("    (no solver failures anywhere in this sweep)")


MIMIC-IV — fixed pre-collapse grid, mean +- sd over 5 seeds

 k  cells  mean_empty_cells  n_seeds_ok   red  red_sd  u_val  u_val_sd  u_str  u_str_sd   syn  syn_sd  worst_sd
 4     32              0.00           5  9.55   10.73  70.99     24.18   0.47      0.61 19.00   13.35     24.18
 5     50              0.00           5 18.84    7.86  49.26     18.21   6.24      6.21 25.66    6.43     18.21
 6     72              0.00           5 23.21    1.72  39.81      7.56   5.16      2.42 31.82    4.36      7.56
 7     98              0.00           5 23.97    1.51  42.24      4.53   3.56      2.48 30.23    1.77      4.53
 8    128              0.00           5 22.89    3.03  39.34      8.20   6.13      3.33 31.63    2.54      8.20
10    200              2.00           5 24.73    1.36  35.70      6.70   7.07      3.27 32.50    4.20      6.70
13    338              7.40           5 24.54    2.12  39.78      2.65   4.16      2.02 31.51    2.71      2.71
16    512             10.20           4 24

## 3. What does collapse_for_occupancy do?

In [4]:
cmp = (sweep.pivot_table(index=["db","k"], columns="arm", values=["u_val","syn","worst_sd"])
       .reset_index())
cmp.columns = ["db","k","syn_collapse","syn_fixed","u_val_collapse","u_val_fixed",
               "sd_collapse","sd_fixed"]
cmp = cmp[["db","k","u_val_fixed","u_val_collapse","syn_fixed","syn_collapse","sd_fixed","sd_collapse"]]
print("fixed vs collapsed, same k-means grid, only the occupancy gate differs\n")
print(cmp.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

diverge = cmp[(cmp.syn_fixed - cmp.syn_collapse).abs() > 1.0]
k_first = int(diverge.k.min()) if len(diverge) else None
print(f"\n  the two arms are identical while the occupancy gate never engages, and")
print(f"  first diverge by more than 1 synergy point at k={k_first}")
worse = cmp[cmp.sd_collapse > cmp.sd_fixed]
print(f"  collapse gives a LARGER seed spread than the fixed grid in "
      f"{len(worse)}/{len(cmp)} (db,k) combinations")
hi = cmp[cmp.k >= 13]
print(f"  at k>=13 collapse drives u_val up by a mean of "
      f"{(hi.u_val_collapse - hi.u_val_fixed).mean():+.1f} points and synergy down by "
      f"{(hi.syn_collapse - hi.syn_fixed).mean():+.1f}")

fixed vs collapsed, same k-means grid, only the occupancy gate differs

      db  k  u_val_fixed  u_val_collapse  syn_fixed  syn_collapse  sd_fixed  sd_collapse
MIMIC-IV  4        70.99           70.99      19.00         19.00     24.18        24.18
MIMIC-IV  5        49.26           49.26      25.66         25.66     18.21        18.21
MIMIC-IV  6        39.81           39.81      31.82         31.82      7.56         7.56
MIMIC-IV  7        42.24           44.76      30.23         30.13      4.53         2.85
MIMIC-IV  8        39.34           37.08      31.63         31.63      8.20        20.92
MIMIC-IV 10        35.70           49.25      32.50         22.99      6.70        26.93
MIMIC-IV 13        39.78           62.65      31.51         17.86      2.71        18.24
MIMIC-IV 16        42.40           97.82      30.25          1.29      3.57         1.66
MIMIC-IV 20        36.83           86.92      33.89          4.49      4.04        16.28
eICU-CRD  4        57.16           57.

## 4. Save

In [5]:
prev = json.load(open("../results/k_resolution_headline_impact.json"))
k13 = fx[fx.k == 13]

payload = {
    "question": "Where do the PID atoms converge in k, and what does collapse_for_occupancy do to them?",
    "provenance": ("Rebuilt by notebooks/07_k_resolution.ipynb on 2026-09-01. The original was "
                   "generated ad hoc on 2026-08-22, its script was not kept, and it ran MIMIC's "
                   "own core panel against the since-retired k=7 headline. This rebuild runs the "
                   "matched 12-analyte panel on BOTH databases at the headline's 5 seeds, so it "
                   "justifies the protocol actually in use."),
    "design": (f"BROJA, mortality, % of I_joint, mean +- sd over {len(SEEDS)} k-means seeds. "
               f"'fixed' = pre-collapse k x k grid; 'collapse' = the same grid with "
               f"collapse_for_occupancy applied. Seeds whose solver cascade fails are excluded "
               f"from the mean and counted in n_seeds_failed."),
    "k_grid": K_GRID,
    "seeds": SEEDS,
    "sweep": rows,
    "why_k13": {
        "criterion": ("Seed spread stops narrowing materially and the solver still converges on "
                      "every seed. The criterion never reads the reported statistic -- choosing k "
                      "by whichever value maximises syn% would be selection bias, since plug-in MI "
                      "rises with resolution."),
        "worst_sd_by_k": {db: {int(r.k): (None if np.isnan(r.worst_sd) else round(r.worst_sd, 3))
                               for _, r in fx[fx.db == db].iterrows()} for db in fx.db.unique()},
        "atoms_at_k13": {r.db: {a: round(r[a], 3) for a in ATOMS} for _, r in k13.iterrows()},
        "degeneracy_above_k13": [
            {"db": r.db, "k": int(r.k), "n_seeds_failed": int(r.n_seeds_failed),
             "mean_empty_cells": round(r.mean_empty_cells, 1)}
            for _, r in fx[fx.n_seeds_failed > 0].iterrows()],
    },
    "collapse_verdict": {
        "first_k_diverging_by_1pt_synergy": k_first,
        "n_combinations_where_collapse_has_larger_seed_spread": int(len(worse)),
        "n_combinations_total": int(len(cmp)),
        "mean_u_val_shift_at_k_ge_13": round(float((hi.u_val_collapse - hi.u_val_fixed).mean()), 2),
        "mean_syn_shift_at_k_ge_13": round(float((hi.syn_collapse - hi.syn_fixed).mean()), 2),
        "conclusion": ("collapse_for_occupancy was written to help the exponential-cone solver on "
                       "sparse joint cells. On this panel it instead drives unique-to-value up and "
                       "synergy down, increasingly with k, and widens rather than narrows the seed "
                       "spread. It is not applied in any headline result."),
    },
    "recommendations": prev["recommendations"],
    "superseded_sections": ("The original's headline_vs_converged and "
                            "published_sensitivity_grid_already_showed_this blocks compared "
                            "against the retired k=7 / seed-0 / occupancy-collapsed headline. "
                            "That protocol is archived under archive/v2_k7_era and the comparison "
                            "is no longer load-bearing, so it is not reproduced here."),
}
out = Path("../results/k_resolution_headline_impact.json")
out.write_text(json.dumps(payload, indent=2, default=float))
print(f"wrote {out}  ({len(rows)} sweep records)")

wrote ../results/k_resolution_headline_impact.json  (36 sweep records)
